# GRU Model Training for WESAD Dataset

This notebook trains a GRU model to classify stress, amusement, and baseline states using the WESAD dataset.

In [9]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import glob

## 1. Configuration and Setup

In [10]:
# Configuration
DATASET_PATH = '../../Dataset/WESAD'
TARGET_USERS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
TARGET_LABELS = {1: 0, 2: 1, 3: 2} # 1: baseline, 2: stress, 3: amusement -> 0, 1, 2
N_FEATURES = 6 # ACC (3), BVP (1), EDA (1), TEMP (1)
DOWNSAMPLE_RATE = 4 # Hz, the rate to downsample all signals to
WINDOW_SIZE_SEC = 60 # seconds
STRIDE_SEC = 1 # seconds

# GRU Hyperparameters
HIDDEN_DIM = 128
LAYER_DIM = 2
OUTPUT_DIM = len(TARGET_LABELS)
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

# Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda:0


## 2. Data Loading and Preprocessing

In [11]:
def load_and_preprocess_data(subject_path):
    """Loads a single subject's data, downsamples, and synchronizes."""
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # --- Extract Wrist Data --- #
    wrist_data = data['signal']['wrist']
    acc = wrist_data['ACC']
    bvp = wrist_data['BVP']
    eda = wrist_data['EDA']
    temp = wrist_data['TEMP']
    labels = data['label']

    # --- Downsample --- #
    # Original sampling rates: ACC=32Hz, BVP=64Hz, EDA=4Hz, TEMP=4Hz, label=700Hz
    acc_down = acc[::32 // DOWNSAMPLE_RATE]
    bvp_down = bvp[::64 // DOWNSAMPLE_RATE]
    eda_down = eda[::4 // DOWNSAMPLE_RATE]
    temp_down = temp[::4 // DOWNSAMPLE_RATE]
    
    # Align labels by resampling at the data points' timestamps
    label_timestamps = np.arange(0, len(labels) / 700.0, 1/700.0)
    data_timestamps = np.arange(0, len(acc_down) / DOWNSAMPLE_RATE, 1/DOWNSAMPLE_RATE)
    idx = np.searchsorted(label_timestamps, data_timestamps, side='left')
    idx = np.clip(idx, 0, len(labels) - 1)
    labels_down = labels[idx].astype(int)
    
    # Find the minimum length to truncate all signals
    min_len = min(len(acc_down), len(bvp_down), len(eda_down), len(temp_down), len(labels_down))
    
    # --- Combine Features --- #
    # Shape: (min_len, N_FEATURES)
    features = np.concatenate([
        acc_down[:min_len],
        bvp_down[:min_len],
        eda_down[:min_len],
        temp_down[:min_len]
    ], axis=1)
    
    labels_final = labels_down[:min_len]
    
    return features, labels_final

def create_windows(features, labels):
    """Creates sliding windows of data and corresponding labels."""
    window_samples = WINDOW_SIZE_SEC * DOWNSAMPLE_RATE
    stride_samples = STRIDE_SEC * DOWNSAMPLE_RATE

    X, y = [], []
    for i in range(0, len(features) - window_samples, stride_samples):
        window_features = features[i : i + window_samples]
        window_labels = labels[i : i + window_samples]
        
        # Use the most frequent label in the window as the segment's label
        most_frequent_label = np.bincount(window_labels).argmax()
        
        if most_frequent_label in TARGET_LABELS:
            X.append(window_features)
            y.append(TARGET_LABELS[most_frequent_label])

    return np.array(X), np.array(y)

In [12]:
# --- Process all subjects --- #
all_X, all_y = [], []
for user in TARGET_USERS:
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X, y = create_windows(features, labels)
        all_X.append(X)
        all_y.append(y)

X_combined = np.concatenate(all_X, axis=0)
y_combined = np.concatenate(all_y, axis=0)

print(f'\nTotal windows created: {len(X_combined)}')
print(f'Feature shape: {X_combined.shape}')
print(f'Label distribution: {np.bincount(y_combined)}')

# --- Split Data FIRST (Data Leakage 방지) --- #
# Train/Val/Test = 70% / 15% / 15%
X_train_unshaped, X_temp_unshaped, y_train, y_temp = train_test_split(
    X_combined, y_combined, test_size=0.3, random_state=42, stratify=y_combined
)

# Val/Test = 50% / 50% (of temp 30%)
X_val_unshaped, X_test_unshaped, y_val, y_test = train_test_split(
    X_temp_unshaped, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'\nTrain set size: {len(X_train_unshaped)}')
print(f'Val set size: {len(X_val_unshaped)}')
print(f'Test set size: {len(X_test_unshaped)}')

# --- Normalize Features (훈련 데이터로만 fit) --- #
# Reshape for scaler: (num_samples * window_length, num_features)
scaler = StandardScaler()

# 훈련 데이터를 reshape하여 fit
X_train_reshaped = X_train_unshaped.reshape(-1, N_FEATURES)
scaler.fit(X_train_reshaped)  # ✓ 훈련 데이터로만 fit

# 훈련, 검증, 테스트 데이터 모두에 transform 적용
X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
X_val_reshaped = X_val_unshaped.reshape(-1, N_FEATURES)
X_val_scaled_reshaped = scaler.transform(X_val_reshaped)
X_test_reshaped = X_test_unshaped.reshape(-1, N_FEATURES)
X_test_scaled_reshaped = scaler.transform(X_test_reshaped)

# 원래 shape로 복원
X_train = X_train_scaled_reshaped.reshape(X_train_unshaped.shape)
X_val = X_val_scaled_reshaped.reshape(X_val_unshaped.shape)
X_test = X_test_scaled_reshaped.reshape(X_test_unshaped.shape)

Processing S2...
Processing S3...
Processing S4...
Processing S5...
Processing S6...
Processing S7...
Processing S8...
Processing S9...
Processing S10...
Processing S11...
Processing S13...
Processing S14...
Processing S15...
Processing S16...
Processing S17...

Total windows created: 33140
Feature shape: (33140, 240, 6)
Label distribution: [17605  9963  5572]

Train set size: 23198
Val set size: 4971
Test set size: 4971


## 3. PyTorch Dataset and DataLoader

In [13]:
class WesadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = WesadDataset(X_train, y_train)
val_dataset = WesadDataset(X_val, y_val)
test_dataset = WesadDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. GRU Model Definition

In [14]:
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # Use nn.GRU (Gated Recurrent Unit)
        # GRU is simpler and faster than LSTM but can capture similar patterns
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.3)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device).requires_grad_()
        
        # GRU forward pass
        out, hn = self.gru(x, h0.detach())
        
        # Index hidden state of last time step
        out = self.fc(out[:, -1, :]) 
        return out

## 5. Model Training

In [15]:
model = GRUModel(N_FEATURES, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (sequences, labels) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * sequences.size(0)
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

Starting training...
Epoch [1/20], Loss: 0.5807, Accuracy: 75.49%
Epoch [2/20], Loss: 0.2326, Accuracy: 90.37%
Epoch [3/20], Loss: 0.1775, Accuracy: 93.69%
Epoch [4/20], Loss: 0.0689, Accuracy: 97.74%
Epoch [5/20], Loss: 0.0625, Accuracy: 98.01%
Epoch [6/20], Loss: 0.0386, Accuracy: 98.88%
Epoch [7/20], Loss: 0.0209, Accuracy: 99.37%
Epoch [8/20], Loss: 0.0412, Accuracy: 98.91%
Epoch [9/20], Loss: 0.0096, Accuracy: 99.68%
Epoch [10/20], Loss: 0.0403, Accuracy: 98.80%
Epoch [11/20], Loss: 0.0079, Accuracy: 99.75%
Epoch [12/20], Loss: 0.0183, Accuracy: 99.47%
Epoch [13/20], Loss: 0.0064, Accuracy: 99.80%
Epoch [14/20], Loss: 0.0030, Accuracy: 99.93%
Epoch [15/20], Loss: 0.0217, Accuracy: 99.38%
Epoch [16/20], Loss: 0.0274, Accuracy: 99.25%
Epoch [17/20], Loss: 0.0028, Accuracy: 99.95%
Epoch [18/20], Loss: 0.0018, Accuracy: 99.97%
Epoch [19/20], Loss: 0.0191, Accuracy: 99.35%
Epoch [20/20], Loss: 0.0021, Accuracy: 99.96%


## 6. Model Evaluation

In [16]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f'\nTest Accuracy: {accuracy:.2f} %')

# Optional: Print classification report for more details
try:
    from sklearn.metrics import classification_report
    report = classification_report(all_labels, all_preds, target_names=['baseline', 'stress', 'amusement'])
    print("\nClassification Report:")
    print(report)
except ImportError:
    print("\nPlease install scikit-learn to see the classification report: pip install -U scikit-learn")


Test Accuracy: 99.98 %

Classification Report:
              precision    recall  f1-score   support

    baseline       1.00      1.00      1.00      2641
      stress       1.00      1.00      1.00      1494
   amusement       1.00      1.00      1.00       836

    accuracy                           1.00      4971
   macro avg       1.00      1.00      1.00      4971
weighted avg       1.00      1.00      1.00      4971

